# Static Analysis in Practice — Scanning Jupyter Notebooks with Semgrep

This notebook shows how to:

1. Create a **vulnerable Jupyter notebook** with ML security issues
2. Extract Python code from the notebook
3. Run **Semgrep with custom ML security rules** on the extracted code

This mirrors the *“Jupyter Notebook Scanning”* part of the video.

## 1. Prerequisites

- `ml-security-rules.yml` should already exist (from the custom rules notebook)
- Semgrep should be installed for live scanning (`pip install semgrep`)

We’ll still show expected findings if Semgrep is not available.

In [1]:
import os
import json
import subprocess

print("="*70)
print("JUPYTER NOTEBOOK SCANNING WITH SEMGREP")
print("="*70)

JUPYTER NOTEBOOK SCANNING WITH SEMGREP


## 2. Create a Sample Vulnerable Notebook

We simulate a notebook that contains:
- Unsafe `pickle.load()`
- Hardcoded API keys
- CSV loading without validation

This is very close to what real data science notebooks look like in the wild.

In [2]:
notebook_content = {
    "cells": [
        {
            "cell_type": "code",
            "source": [
                "# Cell 1: Import libraries\n",
                "import pickle\n",
                "import pandas as pd\n"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "# Cell 2: Load model (VULNERABLE!)\n",
                "model = pickle.load(open('model.pkl', 'rb'))\n"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "# Cell 3: API Key (VULNERABLE!)\n",
                "API_KEY = 'sk-1234567890abcdef'\n",
                "SECRET = 'api_key_secret123'\n"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "# Cell 4: Load data without validation\n",
                "data = pd.read_csv('data.csv')\n"
            ]
        }
    ],
    "metadata": {},
    "nbformat": 4,
    "nbformat_minor": 5
}

with open('vulnerable_notebook.ipynb', 'w', encoding='utf-8') as f:
    json.dump(notebook_content, f, indent=2)

print("\n✓ Created sample notebook: vulnerable_notebook.ipynb")


✓ Created sample notebook: vulnerable_notebook.ipynb


## 3. Extract Python Code from the Notebook

Semgrep works on source files, so we extract all code cells into a `.py` file.

In a real pipeline, this could be automated for all notebooks in a repo.

In [3]:
notebook_code = []
for cell in notebook_content['cells']:
    if cell['cell_type'] == 'code':
        notebook_code.extend(cell['source'])

notebook_py = '\n'.join(notebook_code)

with open('notebook_extracted.py', 'w', encoding='utf-8') as f:
    f.write(notebook_py)

print("✓ Extracted Python code to: notebook_extracted.py")
print("\n[Extracted Code Preview]\n")
print(notebook_py)

✓ Extracted Python code to: notebook_extracted.py

[Extracted Code Preview]

# Cell 1: Import libraries

import pickle

import pandas as pd

# Cell 2: Load model (VULNERABLE!)

model = pickle.load(open('model.pkl', 'rb'))

# Cell 3: API Key (VULNERABLE!)

API_KEY = 'sk-1234567890abcdef'

SECRET = 'api_key_secret123'

# Cell 4: Load data without validation

data = pd.read_csv('data.csv')



## 4. Helper to Run Semgrep

We reuse the same helper pattern as in the custom rules notebook.

In [4]:
def run_semgrep(rule_file, target_files):
    try:
        cmd = ['semgrep', '--config', rule_file, '--json'] + target_files
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30,
            encoding='utf-8',
            errors='replace'
        )
        if result.returncode in (0, 1):
            try:
                return json.loads(result.stdout)
            except Exception:
                return None
        return None
    except FileNotFoundError:
        print("❌ Semgrep not installed. Install with: pip install semgrep")
        return None
    except Exception as e:
        print(f"Error running Semgrep: {e}")
        return None

## 5. Scan the Extracted Notebook Code with Semgrep

We now run Semgrep using `ml-security-rules.yml` against `notebook_extracted.py`.

We expect to see findings for:
- Unsafe `pickle.load()`
- Hardcoded API keys
- Missing data validation on `pd.read_csv()`


In [5]:
print("\n[Scanning Notebook Code]")

nb_findings = run_semgrep('ml-security-rules.yml', ['notebook_extracted.py'])

if nb_findings and 'results' in nb_findings:
    print(f"\n🔍 Found {len(nb_findings['results'])} issues in notebook_extracted.py:")
    for r in nb_findings['results']:
        rule_id = r['check_id'].split('.')[-1]
        line = r['start']['line']
        severity = r['extra']['severity']
        message = r['extra']['message']
        emoji = "🔴" if severity == "ERROR" else "🟡"
        print(f"  {emoji} Line {line}: {rule_id}")
        print(f"     {message}")
else:
    print("\nNote: Semgrep not installed or no findings parsed.")
    print("\nExpected findings in notebook:")
    print("  • Unsafe pickle.load()")
    print("  • Hardcoded API keys (2 instances)")
    print("  • Missing data validation on pd.read_csv()")


[Scanning Notebook Code]

🔍 Found 2 issues in notebook_extracted.py:
  🔴 Line 9: unsafe-pickle-load
     Unsafe pickle deserialization detected. Pickle can execute arbitrary code. Use safer alternatives like joblib, ONNX, or SafeTensors.
  🟡 Line 19: missing-data-validation
     CSV data loaded without validation. Add file existence checks, size limits, and schema validation.


## 6. Summary

You’ve now seen how to:

- Treat **notebooks as code** for security scanning
- Extract code cells into `.py` files
- Apply **custom ML security rules** with Semgrep

This is a practical way to bring static analysis into real data science workflows.

In [6]:
print("\n" + "="*70)
print("NOTEBOOK SCANNING DEMO COMPLETE")
print("="*70)

# Optional cleanup
for fname in ['vulnerable_notebook.ipynb', 'notebook_extracted.py']:
    if os.path.exists(fname):
        os.remove(fname)
print("\n(Optional) Temporary files cleaned up.")


NOTEBOOK SCANNING DEMO COMPLETE

(Optional) Temporary files cleaned up.
